# PBEL: ESC-50 to 4-class Tiny CNN (int8 TFLite)
Run the cells in order (Runtime > Run all). A CPU runtime is enough; no GPU needed.

## 1. Setup and dataset

In [ ]:
!pip install -q soundfile
!git clone --depth 1 https://github.com/karolpiczak/ESC-50.git
!ls ESC-50/audio | wc -l

## 2. Write the scripts

In [ ]:
%%writefile 1_prepare_dataset.py
"""
Step 1 - Prepare the 4-class dataset from ESC-50.

Usage:  python 1_prepare_dataset.py  [path/to/ESC-50-master]
Output: dataset_logmel.npz, mel_filterbank.npy, feature_config.json

Pipeline: ESC-50 clip (44.1 kHz) -> 16 kHz mono -> 1 s windows
          -> RMS normalisation -> log-Mel (61 frames x 40 bands)
The feature code below is deliberately plain NumPy so it can be re-implemented
identically in ESP32-S3 firmware (same window, FFT size, hop, filterbank, log).
"""
import sys, json
import numpy as np, pandas as pd, soundfile as sf
from scipy.signal import resample_poly

ROOT = sys.argv[1] if len(sys.argv) > 1 else "ESC-50-master"
SR, WIN = 16000, 16000            # 16 kHz, 1 s window
N_FFT, HOP, N_MELS = 512, 256, 40
FMIN, FMAX = 0.0, 8000.0
TARGET_RMS = 0.05                 # every window is level-normalised to this RMS
SEED = 42
CLASSES = ["Normal Ambient", "Vehicle Horn", "Construction / Drilling", "Emergency Siren"]
# ESC-50 has no "drilling" class; power-tool sounds are the closest match.
EVENT_MAP = {"car_horn": 1, "chainsaw": 2, "hand_saw": 2, "siren": 3}
# loud events that are NOT our classes and could be confused with them: leave out of Ambient
EXCLUDE = {"clock_alarm", "church_bells", "fireworks", "glass_breaking"}
EVENT_HOP = WIN // 4              # events: dense windows (0.25 s hop) -> more samples
AMBIENT_PER_CLIP = 2              # ambient: at most 2 random windows per clip
AUG_PER_EVENT = 2                 # train-only: noise-mixed copies of each event window

rng = np.random.default_rng(SEED)

def hz2mel(f): return 2595.0 * np.log10(1.0 + f / 700.0)
def mel2hz(m): return 700.0 * (10.0 ** (m / 2595.0) - 1.0)

def mel_filterbank():
    hz = mel2hz(np.linspace(hz2mel(FMIN), hz2mel(FMAX), N_MELS + 2))
    bins = np.fft.rfftfreq(N_FFT, 1.0 / SR)
    fb = np.zeros((N_MELS, len(bins)), np.float32)
    for i in range(N_MELS):
        lo, c, hi = hz[i], hz[i + 1], hz[i + 2]
        fb[i] = np.maximum(0.0, np.minimum((bins - lo) / (c - lo), (hi - bins) / (hi - c)))
    return fb

FB = mel_filterbank()
HANN = np.hanning(N_FFT).astype(np.float32)
rms = lambda x: float(np.sqrt(np.mean(np.square(x)) + 1e-12))

def logmel(x):
    x = x * (TARGET_RMS / rms(x))
    n = 1 + (len(x) - N_FFT) // HOP                      # 61 frames for 16000 samples
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(n)[:, None]
    power = np.abs(np.fft.rfft(x[idx] * HANN, axis=1)) ** 2
    return np.log(power @ FB.T + 1e-6).astype(np.float32)  # (61, 40)

def load16k(path):
    x, sr = sf.read(path, dtype="float32")
    if x.ndim > 1: x = x.mean(axis=1)
    return resample_poly(x, SR // np.gcd(SR, sr), sr // np.gcd(SR, sr)).astype(np.float32)

def win_starts(n, hop): return list(range(0, n - WIN + 1, hop))

meta = pd.read_csv(f"{ROOT}/meta/esc50.csv")
X, y, fold, aug = [], [], [], []
ambient_pool = []                                       # raw ambient windows (train folds) for noise mixing
event_wave = []                                         # (wave, label, fold) for train-fold augmentation

for _, r in meta.iterrows():
    cat = r["category"]
    if cat in EXCLUDE: continue
    x = load16k(f"{ROOT}/audio/{r['filename']}")
    if len(x) < WIN: continue
    starts = win_starts(len(x), EVENT_HOP if cat in EVENT_MAP else WIN // 2)
    levels = np.array([rms(x[s:s + WIN]) for s in starts])
    if cat in EVENT_MAP:                                # keep only windows that really contain the event
        keep = [s for s, l in zip(starts, levels) if l >= 0.3 * levels.max() and l > 1e-3]
        label = EVENT_MAP[cat]
    else:                                               # ambient: skip digital silence, sample a few windows
        keep = [s for s, l in zip(starts, levels) if l > 1e-4]
        keep = list(rng.permutation(keep)[:AMBIENT_PER_CLIP]) if keep else []
        label = 0
    for s in keep:
        w = x[s:s + WIN]
        X.append(logmel(w)); y.append(label); fold.append(int(r["fold"])); aug.append(0)
        if r["fold"] <= 3:                              # only training folds feed augmentation
            (ambient_pool if label == 0 else event_wave).append((w, label, int(r["fold"])))

# noise-mixed augmentation of EVENT windows, training folds only (SNR 5-20 dB)
pool = [w for w, _, _ in ambient_pool]
for w, label, f in event_wave:
    for _ in range(AUG_PER_EVENT):
        noise = pool[rng.integers(len(pool))]
        snr = rng.uniform(5, 20)
        mixed = w + noise * (rms(w) / rms(noise)) / (10 ** (snr / 20))
        X.append(logmel(mixed)); y.append(label); fold.append(f); aug.append(1)

X = np.stack(X); y = np.array(y); fold = np.array(fold); aug = np.array(aug)
np.savez_compressed("dataset_logmel.npz", X=X, y=y, fold=fold, aug=aug, classes=np.array(CLASSES))
np.save("mel_filterbank.npy", FB)
json.dump(dict(sample_rate=SR, window=WIN, n_fft=N_FFT, hop=HOP, n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
               target_rms=TARGET_RMS, frames=int(X.shape[1]), window_fn="numpy.hanning(512) (symmetric)",
               log="ln(power @ mel_fb.T + 1e-6)"), open("feature_config.json", "w"), indent=2)

print("X shape:", X.shape)
print(f"{'class':26s}" + "".join(f"fold{f:>6d}" for f in range(1, 6)) + "   augmented")
for k, name in enumerate(CLASSES):
    print(f"{name:26s}" + "".join(f"{int(((y == k) & (fold == f) & (aug == 0)).sum()):>10d}" for f in range(1, 6))
          + f"{int(((y == k) & (aug == 1)).sum()):>12d}")
print("Saved dataset_logmel.npz, mel_filterbank.npy, feature_config.json")


In [ ]:
%%writefile 2_train_model.py
"""
Step 2 - Train the Tiny CNN, choose the confidence threshold, export int8 TFLite.

Usage:  python 2_train_model.py [epochs]        (default 40)
Needs:  dataset_logmel.npz from step 1
Output: model_int8.tflite, model_data.h, norm.json, confusion_matrix.png, results.txt

Split (ESC-50 folds, no leakage between clips of the same source recording):
    train = folds 1-3 (incl. noise-mixed copies) | validation = fold 4 | test = fold 5
"""
import sys, json
import numpy as np, tensorflow as tf
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

EPOCHS = int(sys.argv[1]) if len(sys.argv) > 1 else 40
tf.keras.utils.set_random_seed(42)
d = np.load("dataset_logmel.npz")
X, y, fold, aug, CLASSES = d["X"], d["y"], d["fold"], d["aug"], list(d["classes"])

tr, va, te = fold <= 3, fold == 4, fold == 5
mean, std = float(X[tr].mean()), float(X[tr].std())      # normalisation from TRAIN only
json.dump(dict(mean=mean, std=std), open("norm.json", "w"))
nx = lambda a: ((a - mean) / std)[..., None].astype(np.float32)
Xtr, Xva, Xte = nx(X[tr]), nx(X[va]), nx(X[te])
ytr, yva, yte = y[tr], y[va], y[te]
cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(4), y=ytr)))

model = tf.keras.Sequential([
    tf.keras.layers.Input(Xtr.shape[1:]),
    *[l for f in (8, 16, 32) for l in (
        tf.keras.layers.Conv2D(f, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling2D(2))],
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(4, activation="softmax"),
])
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=64, class_weight=cw,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)], verbose=2)

out = []
def log(s=""): print(s); out.append(s)

# ---- confidence threshold from VALIDATION set -------------------------------------------
pva = model.predict(Xva, verbose=0)
conf, pred = pva.max(1), pva.argmax(1)
log("\nValidation: confidence threshold vs accuracy on accepted windows")
chosen = 0.5
for th in np.arange(0.50, 0.96, 0.05):
    m = conf >= th
    acc = (pred[m] == yva[m]).mean() if m.any() else float("nan")
    log(f"  th={th:.2f}  accepted={m.mean()*100:5.1f}%  accuracy={acc*100:5.1f}%")
    if m.any() and acc >= 0.95 and chosen == 0.5: chosen = round(float(th), 2)
log(f"Chosen confidence threshold (first with >=95% accepted accuracy, else 0.50): {chosen}")

# ---- TEST set (float model) --------------------------------------------------------------
pte = model.predict(Xte, verbose=0).argmax(1)
log("\nTEST (fold 5) - float model\n" + classification_report(yte, pte, target_names=CLASSES, digits=3))
cm = confusion_matrix(yte, pte); log("Confusion matrix (rows=true, cols=pred):\n" + str(cm))

# ---- int8 quantisation -------------------------------------------------------------------
rep = lambda: (([Xtr[i:i + 1]] for i in np.random.default_rng(0).choice(len(Xtr), 300, replace=False)))
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
blob = conv.convert()
open("model_int8.tflite", "wb").write(blob)
with open("model_data.h", "w") as f:
    f.write("// int8 TFLite model for TensorFlow Lite Micro\nalignas(16) const unsigned char g_model[] = {\n")
    for i in range(0, len(blob), 12): f.write("  " + ", ".join(f"0x{b:02x}" for b in blob[i:i + 12]) + ",\n")
    f.write(f"}};\nconst unsigned int g_model_len = {len(blob)};\n")

it = tf.lite.Interpreter(model_content=blob); it.allocate_tensors()
inp, outp = it.get_input_details()[0], it.get_output_details()[0]
s_in, z_in = inp["quantization"]
q = lambda a: np.clip(np.round(a / s_in + z_in), -128, 127).astype(np.int8)
qp = []
for i in range(len(Xte)):
    it.set_tensor(inp["index"], q(Xte[i:i + 1])); it.invoke(); qp.append(it.get_tensor(outp["index"])[0].argmax())
qp = np.array(qp)
log(f"\nTEST (fold 5) - int8 model accuracy: {(qp == yte).mean()*100:.1f}%   (float: {(pte == yte).mean()*100:.1f}%)")
log(f"Model file size: {len(blob)/1024:.1f} KB   Input quant scale/zero-point: {s_in:.6f} / {z_in}")
log(f"Chosen confidence threshold: {chosen}")
open("results.txt", "w").write("\n".join(out))

fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(CLASSES, rotation=30, ha="right"); ax.set_yticklabels(CLASSES)
for i in range(4):
    for j in range(4): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (test fold)")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=200)
print("Saved model_int8.tflite, model_data.h, norm.json, results.txt, confusion_matrix.png")


## 3. Prepare dataset (a few minutes)

In [ ]:
!python 1_prepare_dataset.py ESC-50

## 4. Train, choose threshold, export model

In [ ]:
!python 2_train_model.py 40 2>&1 | grep -v -E '^(WARNING|I0|W0|E0)'

## 5. Results

In [ ]:
print(open('results.txt').read())
from IPython.display import Image
Image('confusion_matrix.png')

## 6. Download everything

In [ ]:
!zip -q pbel_outputs.zip model_int8.tflite model_data.h norm.json feature_config.json mel_filterbank.npy confusion_matrix.png results.txt
from google.colab import files
files.download('pbel_outputs.zip')